# N111 · DAG动态规划与隐式偏序

**目标：** 按拓扑顺序合并状态并避免循环依赖。

**先修：** N082, N098, N080。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 有向无环；最长路；路径计数；网格递增；记忆化DFS。

**配套讲解来源：** [同名逐章意见](../../comment/111_dag_dp_and_longest_paths.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章的问题都藏在一张"有向无环图（DAG）"里：点与点之间有先后依赖（先修关系、数值递增、边的方向），你要沿着依赖方向找"最长能走多远"。三个代表问题：

1. **矩阵中的最长递增路径**：网格里每步可以上下左右走，但只能走到**数值严格更大**的格子，求最多能走几个格子。（LeetCode 329）
2. **有向图中最大颜色值**：每个节点染一个颜色，沿边走一条路径，路径上同一颜色出现的最多次数就是这条路径的"颜色值"，求所有路径里最大的；图里有环则返回 −1。（LeetCode 1857）
3. **并行课程 III**：n 门课各有时长和先修关系，可以无限并行地同时上多门课，问修完所有课最少要几个月。（LeetCode 2050）

先看具体到数字的例子（“运行示例”部分用的就是它）：

- 输入矩阵 `[[9,9,4],[6,6,8],[2,1,1]]`，即

| 行 | 高度 |
|---|---|
| 0 | 9 9 4 |
| 1 | 6 6 8 |
| 2 | 2 1 1 |

- 输出 `4`。
- 为什么是 4？最长的一条严格递增路是 1 → 2 → 6 → 9：从左下角 (2,1) 出发向左到 (2,0)，向上到 (1,0)，再向上到 (0,0)。每一步数值都严格变大，共 4 个格子。

同一个 cell 还演示了并行课程：3 门课，课 1（3 个月）和课 2（2 个月）没有先修、课 3（5 个月）要求先修 1 和 2。答案是 `8`：第 0 个月同时开课 1 和课 2，课 1 到第 3 个月修完（课 2 第 2 个月就完了，但课 3 要等两个都完成），第 3 个月开课 3，第 8 个月全部结束。瓶颈永远是"最重的那条先修链"。

## 2. 基础知识：先读懂这些词

- **DAG（有向无环图）**：边有方向、顺着走不可能回到起点的图。"无环"是本章一切算法的前提：有环意味着"A 依赖 B、B 又依赖 A"，谁也没法先算，DP 直接失效。
- **隐式偏序 / 隐式图**：矩阵那题的图并没有真的画出来——每个格子是一个点，若相邻格数值严格更大就连一条"从小到大"的边。因为每条边都让数值严格上升，而数值不可能一圈绕回来变大，所以这张隐式图天然无环。这就是"严格递增约束自动排除环"的含义。
- **拓扑序（topological order）**：把所有点排成一列，使每条边都从排在前面的点指向排在后面的点。它就是"依赖关系的合法完成顺序"：前驱都完成了，轮到你才算。
- **入度（indegree）与 Kahn 算法**：入度 = 指向这个点的边数。Kahn 算法反复做一件事：把当前入度为 0 的点（没有任何未完成的前驱）拿出来，再把它们指向的点的入度各减一。拿出来的顺序就是拓扑序；如果拿到最后还有点没被拿走，说明有环。
- **按拓扑序"松弛"**：DP 转移就是"一个点的答案 = 它所有前驱答案的组合（取 max 或累加）"。按拓扑序处理，就能保证轮到某个点时它的前驱全都算完了。注意它不需要 Dijkstra——Dijkstra 是给带权图按距离排序用的，DAG 的无环性已经白送了依赖顺序。
- **逐点携带完整状态向量（26 种颜色各自的最大次数）**：颜色题的关键设计。每个点的 DP 值不是一个数，而是 26 个数——"到这个点为止的路径上，颜色 a 最多出现几次、颜色 b 最多几次……"。不能只存"当前最优颜色"，因为不同后继需要的是不同颜色的统计（推导段的原话：不同后继可能需要不同颜色状态）。
- **关键路径 / 最长路**：并行课程题里，"修完所有课的最短时间"等于"先修链上时间之和最大的那条链"。允许无限并行时，谁也拖不过自己那条最长的先修链。

## 3. 思路推导：从小例子开始

### 主线：矩阵最长递增路，在 3×3 例子上手算（Kahn 分层版）

- **Step 1：把格子看成点、上升看成边。** 格子 A 到相邻格子 B 有边，当且仅当 B 的数值严格更大。
- **Step 2：算每个点的入度。** 入度 = 数值比自己**小**的邻居个数（小格子会连边指过来）。逐格数一遍：

| 格子 | 值 | 更小的邻居 | 入度 |
|---|---|---|---|
| (0,0) | 9 | (1,0)=6 | 1 |
| (0,1) | 9 | (1,1)=6, (0,2)=4 | 2 |
| (0,2) | 4 | 无 | 0 |
| (1,0) | 6 | (2,0)=2 | 1 |
| (1,1) | 6 | (2,1)=1 | 1 |
| (1,2) | 8 | (0,2)=4, (2,2)=1, (1,1)=6 | 3 |
| (2,0) | 2 | (2,1)=1 | 1 |
| (2,1) | 1 | 无 | 0 |
| (2,2) | 1 | 无 | 0 |

- **Step 3：分层剥洋葱。** 第 1 层拿走入度 0 的三个"局部最小"格子：(0,2)=4、(2,1)=1、(2,2)=1。拿走后把它们指向的大格子的入度减一：(1,1) 变 0、(2,0) 变 0、(0,1) 变 1、(1,2) 从 3 减到 1。
- **Step 4：继续剥。** 第 2 层拿 (1,1)=6 和 (2,0)=2 → (0,1)、(1,2)、(1,0) 的入度先后归零。第 3 层拿 (0,1)=9、(1,2)=8、(1,0)=6 → (0,0) 归零。第 4 层拿 (0,0)=9。剥完共 **4 层**，返回 4。
- **Step 5：为什么层数 = 最长路？** 一个格子在第几层被拿走，取决于它"最小的前驱链"有多长：链上第 k 个格子必须等前面 k−1 个全被拿走才轮到它。所以最大层数恰好等于最长递增路的格子数。这条路径就是 1(2,1) → 2(2,0) → 6(1,0) → 9(0,0)，四个格子分属第 1、2、3、4 层。

### 副线一：颜色值 `'abaca'`、边 `0→1, 0→2, 2→3, 3→4`

- 每个点带一个 26 格的小账本，`dp[u][c]` = 到 u 为止的路径上颜色 c 最多出现几次。
- 拓扑序处理（这里 0,1,2,3,4 天然有序）：处理点 u 时，账本已经从所有前驱那里各抄了一遍最大值，然后给自己颜色的格子 +1。
  - 点 0（'a'）：账本 a=1。
  - 点 1（'b'）：抄 0 的账本（a=1），自己 b=1。
  - 点 2（'c'）：抄 0 的账本（a=1），自己 c=1。
  - 点 3（'a'）：抄 2 的账本（a=1,c=1），自己 a=2。
  - 点 4（'a'）：抄 3 的账本（a=2,c=1），自己 a=3。
- 全局最大是点 4 的 a=3，对应路径 0→2→3→4 = 颜色 "a,c,a,a"，三个 a。答案 3。

### 副线二：并行课程 `(3,[(1,3),(2,3)],[3,2,5])`

- `end[v]` = 修完课 v 的最早时刻。没有先修的课（1 和 2）end 就是自己的时长（3 和 2）。
- 课 3 的先修是 1 和 2，它必须等两者都完成：end[3] = max(end[1], end[2]) + 5 = max(3,2)+5 = 8。
- 答案是所有 end 的最大值 8——总工期由"最重的先修链"1→3 决定，课 2 提前修完也没用，这叫关键路径。

## 4. 核心代码：longest_increasing_path

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def longest_increasing_path(matrix):
    if not matrix or not matrix[0]:
        return 0
    m, n = (len(matrix), len(matrix[0]))
    indegree = [[0] * n for _ in range(m)]

    def neighbors(r, c):
        for x, y in [(r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)]:
            if 0 <= x < m and 0 <= y < n:
                yield (x, y)
    for r in range(m):
        for c in range(n):
            indegree[r][c] = sum((matrix[x][y] < matrix[r][c] for x, y in neighbors(r, c)))
    q = deque(((r, c) for r in range(m) for c in range(n) if indegree[r][c] == 0))
    layers = 0
    while q:
        layers += 1
        for _ in range(len(q)):
            r, c = q.popleft()
            for x, y in neighbors(r, c):
                if matrix[x][y] > matrix[r][c]:
                    indegree[x][y] -= 1
                    if indegree[x][y] == 0:
                        q.append((x, y))
    return layers
```

### 逐段读懂代码

### 1. `longest_increasing_path`

（对应程序见下方分段实现与完整代码。）

- 第一行挡空矩阵（`[]` 或 `[[]]`）返回 0。
- `neighbors` 是个生成器，yield 出 (r,c) 的上下左右四个还在界内的邻居；`0<=x<m and 0<=y<n` 把越界的挡掉。
- 双重循环给每个格子算入度：数严格比自己小的邻居有几个（`matrix[x][y]<matrix[r][c]` 是布尔值，`sum` 把 True 当 1 累加）。
- 队列 `q` 一开始装所有入度 0 的"谷底"格子。主循环是分层版 Kahn：`layers+=1` 表示新剥一层；`for _ in range(len(q))` 只处理"这一层开始时"队列里已有的格子——处理过程中新入队的属于下一层，这样 `layers` 才忠实数出层数。
- 内层：拿走 (r,c) 后，对所有数值更大的邻居（也就是它指向的点）入度减一；减到 0 说明这个格子的所有前驱都被拿走了，入队等待下一层。
- 返回层数，理由见第三节 Step 5。注意它数的是"格子的个数"而不是"步数"（1 个格子也是 1 层），和题目口径一致。

### 2. `largest_path_value`

```python
def largest_path_value(colors, edges):
    n = len(colors)
    adj = [[] for _ in range(n)]
    degree = [0] * n
    for u, v in edges:
        adj[u].append(v)
        degree[v] += 1
    dp = [[0] * 26 for _ in range(n)]
    q = deque((i for i, d in enumerate(degree) if d == 0))
    visited = best = 0
    while q:
        u = q.popleft()
        visited += 1
        dp[u][ord(colors[u]) - 97] += 1
        best = max(best, max(dp[u]))
        for v in adj[u]:
            for c in range(26):
                dp[v][c] = max(dp[v][c], dp[u][c])
            degree[v] -= 1
            if degree[v] == 0:
                q.append(v)
    return best if visited == n else -1
```

- 建邻接表、算入度，和上一题同款。
- `dp[u]` 是 26 个数：`ord(colors[u])-97` 把小写字母换算成 0..25（'a' 的编码是 97）。
- 弹出 u 时：`visited+=1` 记账（用来最后判环）；先给自己的颜色 +1（此时 dp[u] 已经合并完所有前驱的账本）；再用 `max(dp[u])` 试着刷新全局答案 `best`。
- 对每个后继 v：把 u 的账本逐颜色抄给 v（`dp[v][c]=max(dp[v][c],dp[u][c])`——每个前驱都抄一遍、取最大，多个前驱就这样汇合）；入度减一，归零就入队。弹出 v 时它的账本已经收齐了所有前驱的信息。
- 最后 `visited==n` 说明 n 个点都处理了（无环）；否则有环，按题意返回 −1。

### 3. `minimum_course_time`

```python
def minimum_course_time(n, relations, time):
    adj = [[] for _ in range(n)]
    degree = [0] * n
    end = list(time)
    for u, v in relations:
        adj[u - 1].append(v - 1)
        degree[v - 1] += 1
    q = deque((i for i, d in enumerate(degree) if d == 0))
    visited = 0
    while q:
        u = q.popleft()
        visited += 1
        for v in adj[u]:
            end[v] = max(end[v], end[u] + time[v])
            degree[v] -= 1
            if degree[v] == 0:
                q.append(v)
    if visited != n:
        raise ValueError('cyclic prerequisites')
    return max(end, default=0)
```

- `end=list(time)`：每门课的最早完成时刻先粗略设成自己的时长（适用于没有先修的课）。
- `adj[u-1].append(v-1)`：题目给的是 1-based 编号，代码里换成 0-based。
- 转移 `end[v]=max(end[v],end[u]+time[v])`：课 v 的每个前驱 u 修完后 v 才能开工，所以 v 的完成时刻至少是"u 完成时刻 + v 自己的时长"；对每个前驱都这样取最大。注意 `end[u]` 此时已经定稿（u 比 v 先出队），这就是拓扑序的用处。
- 收尾 `if visited!=n: raise`：有环（先修关系矛盾）就明确报错拒绝，这是练习 2 的主题。`max(end,default=0)` 里 `default=0` 挡住 n=0 的空输入（`max([])` 会抛异常）。

## 5. 分段实现：按顺序运行

**本章接口：** `longest_increasing_path(matrix)`、`largest_path_value(colors, edges)`、`minimum_course_time(n, relations, time)`

### 导入本章使用的库

In [1]:
from collections import deque

### longest_increasing_path

In [2]:
def longest_increasing_path(matrix):
    if not matrix or not matrix[0]:
        return 0
    m, n = (len(matrix), len(matrix[0]))
    indegree = [[0] * n for _ in range(m)]

    def neighbors(r, c):
        for x, y in [(r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)]:
            if 0 <= x < m and 0 <= y < n:
                yield (x, y)
    for r in range(m):
        for c in range(n):
            indegree[r][c] = sum((matrix[x][y] < matrix[r][c] for x, y in neighbors(r, c)))
    q = deque(((r, c) for r in range(m) for c in range(n) if indegree[r][c] == 0))
    layers = 0
    while q:
        layers += 1
        for _ in range(len(q)):
            r, c = q.popleft()
            for x, y in neighbors(r, c):
                if matrix[x][y] > matrix[r][c]:
                    indegree[x][y] -= 1
                    if indegree[x][y] == 0:
                        q.append((x, y))
    return layers

### largest_path_value

In [3]:
def largest_path_value(colors, edges):
    n = len(colors)
    adj = [[] for _ in range(n)]
    degree = [0] * n
    for u, v in edges:
        adj[u].append(v)
        degree[v] += 1
    dp = [[0] * 26 for _ in range(n)]
    q = deque((i for i, d in enumerate(degree) if d == 0))
    visited = best = 0
    while q:
        u = q.popleft()
        visited += 1
        dp[u][ord(colors[u]) - 97] += 1
        best = max(best, max(dp[u]))
        for v in adj[u]:
            for c in range(26):
                dp[v][c] = max(dp[v][c], dp[u][c])
            degree[v] -= 1
            if degree[v] == 0:
                q.append(v)
    return best if visited == n else -1

### minimum_course_time

In [4]:
def minimum_course_time(n, relations, time):
    adj = [[] for _ in range(n)]
    degree = [0] * n
    end = list(time)
    for u, v in relations:
        adj[u - 1].append(v - 1)
        degree[v - 1] += 1
    q = deque((i for i, d in enumerate(degree) if d == 0))
    visited = 0
    while q:
        u = q.popleft()
        visited += 1
        for v in adj[u]:
            end[v] = max(end[v], end[u] + time[v])
            degree[v] -= 1
            if degree[v] == 0:
                q.append(v)
    if visited != n:
        raise ValueError('cyclic prerequisites')
    return max(end, default=0)

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

matrix=[[9,9,4],[6,6,8],[2,1,1]]
show_table(['行','高度'],list(enumerate(matrix)))
print('严格递增最长路径',longest_increasing_path(matrix))
print('并行课程总时长',minimum_course_time(3,[(1,3),(2,3)],[3,2,5]))

行,高度
0,"[9, 9, 4]"
1,"[6, 6, 8]"
2,"[2, 1, 1]"


严格递增最长路径 4
并行课程总时长 8


## 7. 正确性、适用条件与复杂度

拓扑弹出u时所有前驱已完成，转移可安全取前驱答案的最大值。矩阵Kahn层数等于最长依赖链长度；不是到某个起点的最短距离。课程允许无限并行，开课时间取所有前置完成时间最大值；资源受限调度不是这个模型。

**代价：** 矩阵O(mn)时间空间；颜色DP O(26(V+E))时间O(26V+E)空间；课程O(V+E)时间空间。颜色限小写英文字母，课程关系使用1-based编号。

### 展开理解

**为什么是对的？** 三题共享同一条骨架论证：按 Kahn 顺序出队的点，出队那一刻它的**所有前驱**都已经出队、答案都已经定稿，所以它只管把前驱们的答案按规则合并（取 max、逐颜色取 max、用前驱完成时刻推自己的开工时刻），合并出来的就是自己的最终答案——不存在"后来又有前驱补票"的情况，因为入度归零就是"前驱到齐"的信号。矩阵题的层数论证在第三节 Step 5：格子被拿走的层数等于"以它结尾的最长递增链长度"，因为链上每个格子都要等它前面那个格子被拿走之后入度才会减到位。矩阵图无环的理由是数值严格上升：绕一圈回到起点意味着起点数值严格大于自己，矛盾。颜色题必须带全 26 个颜色而不是一个最优值，是因为"到 u 为止 a 最多 3 次"和"到 u 为止 b 最多 2 次"可能来自不同路径，后继要哪条路取决于自己的颜色，只存一个数就把另一条路的信息丢了。课程题的模型前提是"无限并行、先修关系是唯一的等待理由"，答案等于最长先修链的时间总和；如果机器数量有限、还要抢资源，就不是这个模型了。

**复杂度**：矩阵题 O(mn) 时间和空间（每个格子入队一次、四条边各摸一次）——1000×1000 的矩阵约一百万格子，瞬间跑完。颜色题是 O(26·(V+E)) 时间、O(26·V+E) 空间，26 是常数所以本质上还是线性于图的大小。课程题 O(V+E) 时间空间，n=10⁵ 门课也就是几十万次操作。

## 8. 单元测试：每个用例在检查什么

- `assert longest_increasing_path([[9,9,4],[6,6,8],[2,1,1]])==4`：正常值测试，第三节逐层手算过的例子，检查入度、分层、层数口径（格子数不是步数）都对。
- `assert longest_increasing_path([[7,7],[7,7]])==1`：特殊值测试——全等矩阵里任何相邻格都不满足"严格更大"，一步都走不了，但一个格子本身算长度 1。它专测 `matrix[x][y]<matrix[r][c]` 里那个严格比较：写成 `<=` 会在这条断言上翻车。
- `assert largest_path_value('abaca',[(0,1),(0,2),(2,3),(3,4)])==3`：正常值测试，第三节副线手算的 3（路径 a-c-a-a）。同时它验证了"多前驱账本合并"：点 4 只从点 3 抄账。
- `assert largest_path_value('a',[(0,0)])==-1`：边界测试，自环 (0,0) 是最小的环，那个点永远入度不为 0，`visited` 凑不满 n，必须返回 −1。
- `assert minimum_course_time(3,[(1,3),(2,3)],[3,2,5])==8`：正常值测试，即第一节的 8。注意 1-based 到 0-based 的换算也在被测（写错下标答案就不是 8）。
- 后半部分是随机对拍：`rng=Random(111)` 固定种子；n=1..7，每轮用"只从小编号连向大编号"的随机边生成一个保证无环的 DAG，颜色取 'abc'，时长 1..4。裁判是递归生成器 `paths(u)`：枚举**每一条**从 u 出发的路径（把 u 接到每个后继的每条路径前面）。断言两件事：颜色题的答案等于"所有路径上出现最多的颜色的最大次数"；课程题的答案等于"所有路径时长总和的最大值"——后者正是"无限并行时总工期 = 最长先修链"这条性质的忠实直译。

In [6]:
assert longest_increasing_path([[9, 9, 4], [6, 6, 8], [2, 1, 1]]) == 4

assert longest_increasing_path([[7, 7], [7, 7]]) == 1

assert largest_path_value('abaca', [(0, 1), (0, 2), (2, 3), (3, 4)]) == 3

assert largest_path_value('a', [(0, 0)]) == -1

assert minimum_course_time(3, [(1, 3), (2, 3)], [3, 2, 5]) == 8

from random import Random

from collections import Counter

rng = Random(111)

for n in range(1, 8):
    for _ in range(20):
        edges = [(u, v) for u in range(n) for v in range(u + 1, n) if rng.randrange(3) == 0]
        colors = ''.join((rng.choice('abc') for _ in range(n)))
        times = [rng.randrange(1, 5) for _ in range(n)]
        adj = [[] for _ in range(n)]
        for u, v in edges:
            adj[u].append(v)

        def paths(u):
            yield [u]
            for v in adj[u]:
                for p in paths(v):
                    yield ([u] + p)
        all_paths = [p for u in range(n) for p in paths(u)]
        assert largest_path_value(colors, edges) == max((max(Counter((colors[u] for u in p)).values()) for p in all_paths))
        assert minimum_course_time(n, [(u + 1, v + 1) for u, v in edges], times) == max((sum((times[u] for u in p)) for p in all_paths))

print('N111: 所有本章断言通过')

N111: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 解释矩阵递增约束为什么排除环。

**练习 2：** 带环课程输入明确拒绝。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（解释矩阵递增约束为什么排除环）**：提示——用反证法。假设存在一个环，沿环走一圈每一步数值都严格变大，回到起点时得到"起点数值 > 起点数值"，矛盾。再拿 `[[1,2],[2,1]]` 手算一下四条边的方向（1→2 有两条、另外两条 2↔2 不连），体会"严格"二字去掉后 `[[1,1],[1,1]]` 为什么就有环了。最后可以用 `[[1,2,3],[6,5,4],[7,8,9]]` 这种蛇形矩阵让 `longest_increasing_path` 给出 9 来验证直觉。
- **练习 2（带环课程输入明确拒绝）**：提示——本章实现里 `if visited!=n: raise ValueError('cyclic prerequisites')` 就是拒绝点。你要解释的是"为什么 visited 数不满就是有环"：有环就意味着环上每个点的入度永远等不到归零（环上的点互为因果），它们永远进不了队列；反过来无环时 Kahn 一定能拿光所有点。手算例子建议 `relations=[(1,2),(2,1)]`（两门课互为先修）：两个点入度都是 1，队列一开始就是空的，visited=0 ≠ 2。再说说这个 raise 和颜色题返回 −1 两种"拒绝风格"的差别（抛异常 vs 返回哨兵值）。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
from collections import deque

def longest_increasing_path(matrix):
    if not matrix or not matrix[0]:
        return 0
    m, n = (len(matrix), len(matrix[0]))
    indegree = [[0] * n for _ in range(m)]

    def neighbors(r, c):
        for x, y in [(r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)]:
            if 0 <= x < m and 0 <= y < n:
                yield (x, y)
    for r in range(m):
        for c in range(n):
            indegree[r][c] = sum((matrix[x][y] < matrix[r][c] for x, y in neighbors(r, c)))
    q = deque(((r, c) for r in range(m) for c in range(n) if indegree[r][c] == 0))
    layers = 0
    while q:
        layers += 1
        for _ in range(len(q)):
            r, c = q.popleft()
            for x, y in neighbors(r, c):
                if matrix[x][y] > matrix[r][c]:
                    indegree[x][y] -= 1
                    if indegree[x][y] == 0:
                        q.append((x, y))
    return layers

def largest_path_value(colors, edges):
    n = len(colors)
    adj = [[] for _ in range(n)]
    degree = [0] * n
    for u, v in edges:
        adj[u].append(v)
        degree[v] += 1
    dp = [[0] * 26 for _ in range(n)]
    q = deque((i for i, d in enumerate(degree) if d == 0))
    visited = best = 0
    while q:
        u = q.popleft()
        visited += 1
        dp[u][ord(colors[u]) - 97] += 1
        best = max(best, max(dp[u]))
        for v in adj[u]:
            for c in range(26):
                dp[v][c] = max(dp[v][c], dp[u][c])
            degree[v] -= 1
            if degree[v] == 0:
                q.append(v)
    return best if visited == n else -1

def minimum_course_time(n, relations, time):
    adj = [[] for _ in range(n)]
    degree = [0] * n
    end = list(time)
    for u, v in relations:
        adj[u - 1].append(v - 1)
        degree[v - 1] += 1
    q = deque((i for i, d in enumerate(degree) if d == 0))
    visited = 0
    while q:
        u = q.popleft()
        visited += 1
        for v in adj[u]:
            end[v] = max(end[v], end[u] + time[v])
            degree[v] -= 1
            if degree[v] == 0:
                q.append(v)
    if visited != n:
        raise ValueError('cyclic prerequisites')
    return max(end, default=0)

# 示例与回归测试
assert longest_increasing_path([[9, 9, 4], [6, 6, 8], [2, 1, 1]]) == 4

assert longest_increasing_path([[7, 7], [7, 7]]) == 1

assert largest_path_value('abaca', [(0, 1), (0, 2), (2, 3), (3, 4)]) == 3

assert largest_path_value('a', [(0, 0)]) == -1

assert minimum_course_time(3, [(1, 3), (2, 3)], [3, 2, 5]) == 8

from random import Random

from collections import Counter

rng = Random(111)

for n in range(1, 8):
    for _ in range(20):
        edges = [(u, v) for u in range(n) for v in range(u + 1, n) if rng.randrange(3) == 0]
        colors = ''.join((rng.choice('abc') for _ in range(n)))
        times = [rng.randrange(1, 5) for _ in range(n)]
        adj = [[] for _ in range(n)]
        for u, v in edges:
            adj[u].append(v)

        def paths(u):
            yield [u]
            for v in adj[u]:
                for p in paths(v):
                    yield ([u] + p)
        all_paths = [p for u in range(n) for p in paths(u)]
        assert largest_path_value(colors, edges) == max((max(Counter((colors[u] for u in p)).values()) for p in all_paths))
        assert minimum_course_time(n, [(u + 1, v + 1) for u, v in edges], times) == max((sum((times[u] for u in p)) for p in all_paths))

print('N111: 所有本章断言通过')

N111: 所有本章断言通过


## 11. 代表题与迁移

- **329. Longest Increasing Path in a Matrix**：练"把网格看成隐式 DAG + Kahn 分层数最长链"，对应 `longest_increasing_path`；记忆化 DFS 是它的另一副面孔，两者等价。
- **1857. Largest Color Value in a Directed Graph**：练"每个点携带完整状态向量（26 色）沿拓扑序合并 + 判环返回 −1"，对应 `largest_path_value`。
- **2050. Parallel Courses III**：练"拓扑序上的关键路径 DP：end[v]=max(end[前驱])+time[v]"，对应 `minimum_course_time`，顺便体会无限并行模型和最长加权链的等价性。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。